# Витрина эквайринга: Excel-отчёт бизнеса + обогащение из lake

База — Excel-отчёт за месяц (1 строка = 1 `agr_id`).
Из lake добавляются: `ssp_ocrm`, `cdi_id`, `ogrn` (по `inn`).

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

In [ ]:
excel_sources = [
    {'report_month': '2026-01-01', 'path': '/home/jovyan/documents/Equaring/Data/01_Январь_2026.xlsx', 'header': 1},
    {'report_month': '2026-02-01', 'path': '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx', 'header': 0},
    {'report_month': '2026-03-01', 'path': '/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx', 'header': 0},
]

output_csv_path = './datamart_month_excel_enriched_q1.csv'
output_monthly_summary_csv_path = './q1_tariff_monthly_summary.csv'
output_unique_tariffs_csv_path = './q1_unique_tariffs_by_month.csv'
output_missing_tariff_agr_csv_path = './q1_agr_id_without_tariff.csv'

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

## 1) Загрузка Excel

In [ ]:
def normalize_inn(value):
    if pd.isna(value):
        return None
    s = re.sub(r'\D+', '', str(value))
    if not s:
        return None
    if len(s) == 9:
        s = '0' + s
    elif len(s) == 11:
        s = '0' + s
    return s


def normalize_colname(value):
    s = str(value).lower().replace('\n', ' ').replace('\r', ' ').replace('\xa0', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    s = s.replace('₽', 'руб').replace('%', 'pct')
    s = re.sub(r'[^a-zа-я0-9]+', '', s)
    return s


def pick_column(columns, aliases):
    norm_map = {normalize_colname(c): c for c in columns}
    for a in aliases:
        k = normalize_colname(a)
        if k in norm_map:
            return norm_map[k]
    return None


frames = []
for src in excel_sources:
    raw = pd.read_excel(src['path'], header=src['header'])

    col_inn    = pick_column(raw.columns, ['ИНН', 'INN'])
    col_agr_id = pick_column(raw.columns, ['ID договора', 'agr_id'])
    col_n_cmp  = pick_column(raw.columns, ['Номер организации', 'n_cmp', 'N_CMP'])
    if col_inn is None or col_agr_id is None:
        raise ValueError(f'Не найдены ИНН/agr_id в {src["path"]}. Колонки: {list(raw.columns)}')

    raw['inn']          = raw[col_inn].apply(normalize_inn)
    raw['agr_id']       = raw[col_agr_id].astype(str).str.strip()
    raw['n_cmp']        = raw[col_n_cmp].astype(str).str.strip() if col_n_cmp else None
    raw['report_month'] = pd.to_datetime(src['report_month'])
    frames.append(raw)

excel_df = pd.concat(frames, ignore_index=True)

print(f'Excel (3 месяца): {len(excel_df):,} строк')
print(f'  по месяцам:')
print(excel_df.groupby('report_month').size())
print(f'Уникальных agr_id: {excel_df["agr_id"].nunique():,}')
print(f'Уникальных inn:    {excel_df["inn"].dropna().nunique():,}')
excel_df.head(5)

## 2) `ssp_ocrm` и `cdi_id` по `inn` (через OCRM/CDI)

In [ ]:
inn_values = sorted(excel_df['inn'].dropna().astype(str).unique().tolist())
inn_in = ', '.join([f"'{x}'" for x in inn_values]) if inn_values else "''"

sql_ocrm = f"""
with ocrm_current as (
  select
    regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') as inn,
    cast(soe.row_id as string) as row_id,
    case
      when soe.x_area_resp like 'ДММБ%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ (ми%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ(ми%' then 'ДМ'
      when soe.x_area_resp like 'ДМСБ (ма%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ(ма%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ (ср%' then 'ДМСБ'
      when soe.x_area_resp like 'ДМСБ(ср%' then 'ДМСБ'
      when soe.x_area_resp like 'ДСБ%' then 'ДМСБ'
      when soe.x_area_resp like 'ДКБ%' then 'ДКБ'
      when soe.x_area_resp like 'ДРПА%' then 'ДРПА'
      when soe.x_area_resp like 'ДРА%' then 'ДРПА'
      when lower(soe.x_area_resp) like 'не под%' then 'Не подлежит сегментации'
      when nullif(trim(cast(soe.x_area_resp as string)), '') is null then null
      else null
    end as ssp_ocrm,
    row_number() over (
      partition by regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '')
      order by cast(soe.created as timestamp) desc,
               cast(soe.row_id as string) desc
    ) as rn
  from ocrm_ul.s_org_ext soe
  where regexp_replace(trim(cast(soe.x_inn as string)), '[^0-9]', '') in ({inn_in})
    and coalesce(soe.x_removed_flg, 'N') = 'N'
    and coalesce(soe.x_duplicate_flg, 'N') = 'N'
),
ocrm_one as (
  select inn, row_id, ssp_ocrm
  from ocrm_current
  where rn = 1
)
select
  o.inn,
  o.ssp_ocrm,
  cast(e.party_id as string) as cdi_id
from ocrm_one o
left join cdiul.ext_id_org e
  on cast(e.cmo_ext_party_source_id as string) = o.row_id
 and upper(cast(e.cmo_ext_source_system as string)) like 'OCRM%'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    ocrm_df = imp.fetch(sql_ocrm)

if ocrm_df is None:
    ocrm_df = pd.DataFrame(columns=['inn', 'ssp_ocrm', 'cdi_id'])

if not ocrm_df.empty:
    ocrm_df['inn']    = ocrm_df['inn'].astype(str)
    ocrm_df['cdi_id'] = ocrm_df['cdi_id'].astype(str)
    allowed = {'ДМ', 'ДМСБ', 'ДКБ', 'ДРПА', 'Не подлежит сегментации'}
    ocrm_df['ssp_ocrm'] = ocrm_df['ssp_ocrm'].where(ocrm_df['ssp_ocrm'].isin(allowed), None)
    ocrm_df = ocrm_df.drop_duplicates(subset=['inn'], keep='first')

print(f'OCRM/CDI: {len(ocrm_df):,} строк')
ocrm_df.head(5)

## 3) `ogrn` по `cdi_id` (CDI -> CFT -> R2)

In [ ]:
cdi_values = sorted(ocrm_df['cdi_id'].dropna().astype(str).unique().tolist())
cdi_values = [x for x in cdi_values if x and x != 'nan']
cdi_in = ', '.join([f"'{x}'" for x in cdi_values]) if cdi_values else "''"

sql_ogrn = f"""
with cft as (
  select
    cast(e.party_id as string) as cdi_id,
    cast(e.cmo_ext_party_source_id as string) as cft_id
  from cdiul.ext_id_org e
  where cast(e.party_id as string) in ({cdi_in})
    and upper(cast(e.cmo_ext_source_system as string)) like 'CFT%'
)
select
  c.cdi_id,
  cast(corp.c_register_gos_reg_num_rec as string) as ogrn
from cft c
left join ods.scd1_z_cl_corp corp
  on cast(corp.id as string) = c.cft_id
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    ogrn_df = imp.fetch(sql_ogrn)

if ogrn_df is None:
    ogrn_df = pd.DataFrame(columns=['cdi_id', 'ogrn'])

if not ogrn_df.empty:
    ogrn_df['cdi_id'] = ogrn_df['cdi_id'].astype(str)
    ogrn_df['ogrn']   = ogrn_df['ogrn'].astype(str)
    ogrn_df = ogrn_df.drop_duplicates(subset=['cdi_id'], keep='first')

print(f'OGRN: {len(ogrn_df):,} строк')
ogrn_df.head(5)

## 4) `mcc` по `n_cmp` (через `scd1_merchants`)

In [ ]:
n_cmp_values = sorted(excel_df['n_cmp'].dropna().astype(str).unique().tolist())
n_cmp_in = ', '.join([f"'{x}'" for x in n_cmp_values]) if n_cmp_values else "''"

sql_mcc = f"""
select
  cast(m.n_cmp as string) as n_cmp,
  cast(max(m.n_mcc) as string) as mcc
from ods_alpha.scd1_merchants m
where cast(m.n_cmp as string) in ({n_cmp_in})
  and coalesce(m.ods_deleted_flg, '0') <> '1'
group by cast(m.n_cmp as string)
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    mcc_df = imp.fetch(sql_mcc)

if mcc_df is None:
    mcc_df = pd.DataFrame(columns=['n_cmp', 'mcc'])

if not mcc_df.empty:
    mcc_df['n_cmp'] = mcc_df['n_cmp'].astype(str)
    mcc_df['mcc']   = mcc_df['mcc'].astype(str)

print(f'MCC: {len(mcc_df):,} строк')
mcc_df.head(5)

## 5) `tariff_name` по `agr_id` (через R2)

In [ ]:
agr_values = sorted(excel_df['agr_id'].dropna().astype(str).unique().tolist())
agr_in = ', '.join([f"'{x}'" for x in agr_values]) if agr_values else "''"

sql_tariff = f"""
select
  cast(m.id as string) as agr_id,
  cast(max(tp.c_name) as string) as tariff_name
from ods.scd1_z_r2_ip_merchants m
left join ods.scd1_z_r2_tariff_plan tp
  on cast(tp.id as string) = cast(m.c_tariff_plan as string)
where cast(m.id as string) in ({agr_in})
group by cast(m.id as string)
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    tariff_df = imp.fetch(sql_tariff)

if tariff_df is None:
    tariff_df = pd.DataFrame(columns=['agr_id', 'tariff_name'])

if not tariff_df.empty:
    tariff_df['agr_id']      = tariff_df['agr_id'].astype(str)
    tariff_df['tariff_name'] = tariff_df['tariff_name'].astype(str)

print(f'Tariff: {len(tariff_df):,} строк')
tariff_df.head(5)

## 6) Финальная сборка: переименование на английский + MVP-набор

In [ ]:
rows_before_merge = len(excel_df)

final_df = excel_df.merge(ocrm_df,   on='inn',    how='left')
final_df = final_df.merge(ogrn_df,   on='cdi_id', how='left')
final_df = final_df.merge(mcc_df,    on='n_cmp',  how='left')
final_df = final_df.merge(tariff_df, on='agr_id', how='left')

# Убираем Excel-колонку 'Тариф', чтобы tariff_name из lake не перезаписался
excel_tariff_col = pick_column(final_df.columns, ['Тариф'])
if excel_tariff_col is not None:
    final_df = final_df.drop(columns=[excel_tariff_col])

column_mapping = {
    'Наименование':              'company_name',
    'Номер договора':            'contract_number',
    'Дата регистрации договора': 'd_valid_from',
    'Дата закрытия договора':    'd_valid_to',
    'Филиал':                    'branch_nm',
    'ВСП договора':              'vsp_name',
    'Код ВСП':                   'vsp_code',
    'Ко-во торговых точек':      'retl_cnt',
    'Кол-во терминалов':         'term_cnt',
    'Количеств операций':        'trx_cnt',
    'Сумма операций':            'trx_sum',
    'Комиссия \n(% с операций)': 'commission_from_ops',
    'Комиссия \n(₽ в месяц)':    'commission_monthly',
    'Комиссия эквайринга':       'commission_total',
    'Комиссия МПС (IRF, ₽)':     'int_component',
    'АУР':                       'aur',
    'Амортизация':               'amortization',
    'ЧОД':                       'chod',
    'Фин. Рез.':                 'fin_result',
}

rename_map = {}
for ru, en in column_mapping.items():
    actual = pick_column(final_df.columns, [ru])
    if actual is not None:
        rename_map[actual] = en

final_df = final_df.rename(columns=rename_map)

# Обрезка branch_nm по первое 'РФ' включительно (как в datamart_month_acquiring.ipynb)
if 'branch_nm' in final_df.columns:
    final_df['branch_nm'] = final_df['branch_nm'].apply(
        lambda v: re.match(r'^(.*?РФ)', str(v)).group(1)
        if pd.notna(v) and 'РФ' in str(v) else v
    )

mvp_columns = [
    'report_month', 'inn', 'agr_id', 'cdi_id', 'ssp_ocrm', 'ogrn',
    'company_name', 'contract_number', 'd_valid_from', 'd_valid_to',
    'branch_nm', 'vsp_name', 'vsp_code', 'tariff_name', 'mcc',
    'retl_cnt', 'term_cnt', 'trx_cnt', 'trx_sum',
    'commission_from_ops', 'commission_monthly', 'commission_total',
    'int_component', 'aur', 'amortization', 'chod', 'fin_result',
]

final_df = final_df[[c for c in mvp_columns if c in final_df.columns]]

# Целые поля: nullable Int64 (без .0 в CSV)
int_cols = ['vsp_code', 'mcc', 'retl_cnt', 'term_cnt', 'trx_cnt']
for col in int_cols:
    if col in final_df.columns:
        final_df[col] = pd.to_numeric(final_df[col], errors='coerce').astype('Int64')

# Денежные поля: округление до 2 знаков
money_cols = ['trx_sum', 'commission_from_ops', 'commission_monthly', 'commission_total',
              'int_component', 'aur', 'amortization', 'chod', 'fin_result']
for col in money_cols:
    if col in final_df.columns:
        final_df[col] = pd.to_numeric(final_df[col], errors='coerce').round(2)

# Нормализация текстового тарифа и месячной метки для расчетов
tariff_series = final_df['tariff_name'] if 'tariff_name' in final_df.columns else pd.Series([None] * len(final_df), index=final_df.index)
report_month_series = pd.to_datetime(final_df['report_month'], errors='coerce')

tariff_clean = tariff_series.where(tariff_series.notna(), None)
tariff_clean = tariff_clean.astype('string').str.strip().replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})

metric_df = pd.DataFrame({
    'report_month': report_month_series,
    'tariff_name': tariff_clean,
})
metric_df = metric_df[metric_df['report_month'].notna()].copy()
metric_df['report_month'] = metric_df['report_month'].dt.strftime('%Y-%m')

# 1) Уникальные названия тарифов по месяцам
unique_tariffs_by_month = (
    metric_df[metric_df['tariff_name'].notna()]
    .drop_duplicates(subset=['report_month', 'tariff_name'])
    .sort_values(['report_month', 'tariff_name'])
    .reset_index(drop=True)
)

# 2) Доля строк с тарифом, содержащим 'акт' (без учета регистра)
metric_df['is_akt_tariff'] = metric_df['tariff_name'].fillna('').str.contains('акт', case=False, regex=False)

monthly_summary = (
    metric_df.groupby('report_month', as_index=False)
    .agg(
        total_rows=('report_month', 'size'),
        unique_tariff_names=('tariff_name', lambda s: s.dropna().nunique()),
        akt_rows=('is_akt_tariff', 'sum'),
        empty_tariff_rows=('tariff_name', lambda s: s.isna().sum()),
    )
    .sort_values('report_month')
    .reset_index(drop=True)
)
monthly_summary['akt_share_pct'] = np.where(
    monthly_summary['total_rows'] > 0,
    (monthly_summary['akt_rows'] / monthly_summary['total_rows'] * 100).round(2),
    0.0,
)
monthly_summary = monthly_summary[[
    'report_month', 'total_rows', 'unique_tariff_names', 'akt_rows', 'akt_share_pct', 'empty_tariff_rows'
]]

# 3) QC: полнота маппинга и сохранение количества строк
rows_after_merge = len(final_df)
missing_tariff_agr = (
    final_df.loc[tariff_clean.isna(), ['report_month', 'agr_id']]
    .dropna(subset=['agr_id'])
    .copy()
)
missing_tariff_agr['report_month'] = pd.to_datetime(missing_tariff_agr['report_month'], errors='coerce').dt.strftime('%Y-%m')
missing_tariff_agr = (
    missing_tariff_agr
    .drop_duplicates(subset=['report_month', 'agr_id'])
    .sort_values(['report_month', 'agr_id'])
    .reset_index(drop=True)
)

print(f'Итого: {rows_after_merge:,} строк, {len(final_df.columns)} колонок')
for col in ['ssp_ocrm', 'cdi_id', 'ogrn', 'mcc', 'tariff_name']:
    if col in final_df.columns:
        print(f'  {col:11s} заполнено: {final_df[col].notna().sum():,}')

print('\nQC:')
print(f'  строк до merge:   {rows_before_merge:,}')
print(f'  строк после merge:{rows_after_merge:,}')
print(f'  строк без tariff_name: {int(tariff_clean.isna().sum()):,} ({(tariff_clean.isna().mean() * 100):.2f}%)')
print(f'  уникальных agr_id без tariff_name: {missing_tariff_agr["agr_id"].nunique():,}')

print('\nМесячный свод тарифов Q1:')
print(monthly_summary)

# 4) Экспорт артефактов
output_path = Path(output_csv_path)
monthly_summary_path = Path(output_monthly_summary_csv_path)
unique_tariffs_path = Path(output_unique_tariffs_csv_path)
missing_tariff_agr_path = Path(output_missing_tariff_agr_csv_path)

final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
monthly_summary.to_csv(monthly_summary_path, index=False, encoding='utf-8-sig')
unique_tariffs_by_month.to_csv(unique_tariffs_path, index=False, encoding='utf-8-sig')
missing_tariff_agr.to_csv(missing_tariff_agr_path, index=False, encoding='utf-8-sig')

print('\nСохранены файлы:')
print(f'  datamart:               {output_path.resolve()}')
print(f'  monthly summary:        {monthly_summary_path.resolve()}')
print(f'  unique tariffs by month:{unique_tariffs_path.resolve()}')
print(f'  missing tariff agr_id:  {missing_tariff_agr_path.resolve()}')

final_df.head(5)